In [ ]:
import os
import sys
import gc
import json
import time
import random
import re
import types
import signal
import contextlib
import subprocess
import importlib.util
import importlib.machinery
from typing import Any, Dict, List, Optional, Tuple

# ============================================================
# Kaggle-safe bootstrap
# ============================================================

def _stub_torchaudio() -> None:
    """
    Some Kaggle images have a broken torchaudio binary that can break
    transformers imports through optional audio dependencies.
    """
    if "torchaudio" in sys.modules:
        return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    submods = ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]
    sys.modules["torchaudio"] = ta
    for sm in submods:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m

_stub_torchaudio()

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])

for mod, pipname in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("sentencepiece", "sentencepiece"),
    ("peft", "peft"),
    ("plotly", "plotly"),
]:
    ensure_pkg(mod, pipname)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "phi3_adapters\\phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_residual_stream_safe_fixed"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
HTML_DIR = os.path.join(OUT_DIR, "html")
REPORT_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
for p in [OUT_DIR, CSV_DIR, PLOT_DIR, HTML_DIR, REPORT_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE_COUNT = torch.cuda.device_count()
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# Exactly 2 examples per benchmark total.
N_GSM8K = 2
N_STRATEGYQA = 2
N_MMLU = 2
MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
    "formal_logic",
    "high_school_mathematics",
]

MAX_NEW_TOKENS = {
    "gsm8k": 96,
    "strategyqa": 24,
    "mmlu": 12,
}

VERBOSE = True

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

def log(msg: str) -> None:
    if VERBOSE:
        print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def gpu_report(prefix: str = "GPU") -> None:
    if not torch.cuda.is_available():
        log("GPU report: CUDA unavailable.")
        return
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)
        log(f"{prefix} {i}: allocated={alloc:.2f} GB reserved={reserved:.2f} GB")

log(f"Detected {DEVICE_COUNT} GPU(s). Running sequentially to avoid OOM.")
gpu_report("Startup GPU")

# ============================================================
# HELPERS
# ============================================================

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

@contextlib.contextmanager
def timer(label: str):
    start = time.time()
    log(f"{label} ...")
    try:
        yield
    except Exception as e:
        log(f"{label} failed after {time.time() - start:.1f}s: {type(e).__name__}: {e}")
        raise
    else:
        log(f"{label} done in {time.time() - start:.1f}s")

@contextlib.contextmanager
def time_limit(seconds: int, label: str):
    if seconds <= 0 or not hasattr(signal, "SIGALRM"):
        yield
        return

    def _handler(signum, frame):
        raise TimeoutError(f"{label} timed out after {seconds}s")

    old_handler = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def save_df(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)
    log(f"Saved CSV -> {path}")

def save_json(obj: Any, path: str) -> None:
    def _default(x):
        if isinstance(x, (np.integer, np.floating)):
            return x.item()
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, (set, tuple)):
            return list(x)
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().tolist()
        return str(x)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_default)
    log(f"Saved JSON -> {path}")

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    log(f"Saved figure -> {path}")

def preview_text(s: str, max_len: int = 110) -> str:
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def sample_dataset(ds, n: int, seed: int, tag: str):
    n = min(n, len(ds))
    cache_path = os.path.join(CACHE_DIR, f"{tag}_n{n}_seed{seed}.json")
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            idx = json.load(f)
        log(f"Loaded cached sample indices for {tag} ({len(idx)} items).")
        return ds.select(idx), idx
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(ds), size=n, replace=False).tolist()
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(idx, f)
    log(f"Sampled {len(idx)} items for {tag}.")
    return ds.select(idx), idx

def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None:
        return None
    try:
        x = float(str(pred).replace(",", ""))
        if abs(x - round(x)) < 1e-8:
            return str(int(round(x)))
        return str(x)
    except Exception:
        return None

def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try:
        return abs(float(pred) - float(gold)) <= 1e-6
    except Exception:
        return False

def extract_number(text: str) -> Optional[str]:
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = span.replace(",", "")
    boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
    if boxed:
        nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
        if nums:
            return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return nums[-1] if nums else None

def extract_yes_no(text: str) -> Optional[str]:
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text: str) -> Optional[str]:
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    up = span.upper().strip()
    if up in ["A", "B", "C", "D"]:
        return up
    for pat in [r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?", r"<ANSWER>\s*\(?([ABCD])\)?", r"\b([ABCD])\b\s*$"]:
        hits = re.findall(pat, up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", up)
    return hits[-1].upper() if hits else None

def explicit_answer_present(text: str) -> bool:
    return bool(re.search(r"\b(yes|no|[ABCD])\b", str(text), flags=re.IGNORECASE))

def normalize_choice_order(choices: List[str], perm: Tuple[int, int, int, int]):
    return [choices[i] for i in perm]

def canonical_letter_from_perm_answer(pred_letter: Optional[str], perm: Tuple[int, int, int, int]) -> Optional[str]:
    if pred_letter is None:
        return None
    pred_letter = pred_letter.upper().strip()
    if pred_letter not in ["A", "B", "C", "D"]:
        return None
    idx = ord(pred_letter) - 65
    return chr(65 + perm[idx])

def get_mmlu_choices(sample: Dict[str, Any]) -> List[str]:
    choices = sample.get("choices")
    if choices is None:
        return []
    if isinstance(choices, str):
        try:
            parsed = json.loads(choices)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    if isinstance(choices, list):
        return choices
    try:
        return list(choices)
    except Exception:
        return []

def safe_mmlu_gold(sample: Dict[str, Any]) -> Optional[str]:
    g = sample.get("gold")
    if g is None:
        return None
    g = str(g).strip().upper()
    return g if g in ["A", "B", "C", "D"] else None

def token_ids_for_text(tokenizer, text: str) -> List[int]:
    forms = [text, f" {text}", f"\n{text}", text.upper(), f" {text.upper()}", text.lower(), f" {text.lower()}"]
    ids = []
    for form in forms:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1:
            ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) > 0:
            ids.append(enc[0])
    return sorted(set(ids))

def topn_token_strings(tokenizer, logits: torch.Tensor, n: int = 5) -> List[Tuple[str, float]]:
    probs = torch.softmax(logits.float(), dim=-1)
    vals, ids = torch.topk(probs, k=min(n, probs.shape[-1]))
    out = []
    for p, i in zip(vals.tolist(), ids.tolist()):
        out.append((tokenizer.decode([i]).strip(), float(p)))
    return out

def candidate_tokens(tokenizer, sample: Dict[str, Any], final_logits: torch.Tensor, pred: Optional[str]) -> List[str]:
    toks = []
    if sample["task"] == "strategyqa":
        toks += ["yes", "no"]
    elif sample["task"] == "mmlu":
        toks += ["A", "B", "C", "D"]
    elif sample["task"] == "gsm8k":
        if sample.get("gold"):
            toks.append(str(sample["gold"]))
        if pred:
            toks.append(str(pred))
    toks += [t for t, _ in topn_token_strings(tokenizer, final_logits, n=5)]
    uniq = []
    for t in toks:
        if t and t not in uniq:
            uniq.append(t)
    return uniq[:8]

def np_pca_2d(X: np.ndarray):
    X = np.asarray(X, dtype=np.float64)
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    Z = Xc @ Vt[:2].T
    return Z, Vt[:2]

def build_task_prompt(sample: Dict[str, Any]) -> str:
    if sample["task"] == "gsm8k":
        return (
            "Solve the math problem carefully.\n"
            "Return reasoning in <think></think> and the final number in <answer></answer>.\n\n"
            f"Question: {sample['question']}\n\n"
            "<think>"
        )
    if sample["task"] == "strategyqa":
        return (
            "Answer the question with yes or no. Use <think></think> if needed.\n\n"
            f"Question: {sample['question']}\n"
            "Answer:"
        )
    if sample["task"] == "mmlu":
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'][:4])])
        return (
            "Choose the correct option and return only a single letter in <answer></answer>.\n"
            "Use <think></think> for reasoning.\n\n"
            f"Question: {sample['question']}\n"
            f"{opts}\n\n"
            "<think>"
        )
    raise ValueError(sample["task"])

def difficulty_score(task: str, question: str) -> float:
    q = str(question).strip()
    if task == "gsm8k":
        ops = sum(q.count(x) for x in ["+", "-", "*", "/", "(", ")"])
        nums = len(re.findall(r"\d+", q))
        return min(1.0, 0.08 * len(q) / 10 + 0.12 * ops + 0.08 * nums)
    if task == "strategyqa":
        markers = sum(1 for w in [" not ", " never ", " except ", " least ", " most ", " all ", " any "] if w in f" {q.lower()} ")
        return min(1.0, 0.07 * len(q) / 10 + 0.18 * markers)
    if task == "mmlu":
        return min(1.0, 0.04 * len(q) / 10)
    return 0.5

def preferred_route(task: str, question: str) -> str:
    d = difficulty_score(task, question)
    if task == "gsm8k":
        return "selfcheck" if d >= 0.55 else "reason"
    if task == "strategyqa":
        return "selfcheck" if d >= 0.50 else "direct"
    if task == "mmlu":
        return "direct"
    return "reason"

def model_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cpu")

# ============================================================
# DATA LOADING
# ============================================================

def load_gsm8k_dataset():
    for repo in ["gsm8k", "openai/gsm8k", "madrylab/gsm8k-platinum"]:
        try:
            log(f"Loading GSM8K from {repo} ...")
            if repo == "gsm8k":
                ds = load_dataset(repo, "main", split="test")
            elif repo == "openai/gsm8k":
                ds = load_dataset(repo, "main", split="test")
            else:
                ds = load_dataset(repo, "main", split="test")
            log(f"Loaded GSM8K ({repo}) with {len(ds)} rows.")
            return ds, repo
        except Exception as e:
            log(f"GSM8K load failed for {repo}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load GSM8K.")

def load_strategyqa_dataset():
    log("Loading StrategyQA test split ...")
    ds = load_dataset("ChilleD/StrategyQA", split="test")
    log(f"Loaded StrategyQA with {len(ds)} rows.")
    return ds

def load_mmlu_subject(subject: str):
    for split in ["test", "validation", "dev"]:
        try:
            log(f"Loading MMLU subject={subject} split={split} ...")
            ds = load_dataset("cais/mmlu", subject, split=split)
            log(f"Loaded MMLU subject={subject} split={split} with {len(ds)} rows.")
            return ds, split
        except Exception as e:
            log(f"MMLU load failed for {subject}/{split}: {type(e).__name__}: {e}")
    raise RuntimeError(f"Could not load MMLU subject {subject}")

# ============================================================
# MODEL LOADING
# ============================================================

def build_max_memory_dict() -> Dict[str, str]:
    if DEVICE_COUNT == 0:
        return {"cpu": "64GiB"}
    out = {}
    for i in range(DEVICE_COUNT):
        total_gib = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        out[i] = f"{max(8, int(total_gib * 0.88))}GiB"
    out["cpu"] = "64GiB"
    return out

MAX_MEMORY = build_max_memory_dict()

def sanitize_rope_scaling(config):
    rs = getattr(config, "rope_scaling", None)
    if isinstance(rs, dict):
        rope_type = str(rs.get("type", "")).strip().lower()
        if rope_type in {"default", "", "none", "null"}:
            config.rope_scaling = None
        else:
            cleaned = dict(rs)
            if "type" in cleaned:
                cleaned["type"] = rope_type
            config.rope_scaling = cleaned
    else:
        config.rope_scaling = None
    return config

def has_adapter(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    files = set(os.listdir(path))
    return any(x in files for x in ["adapter_config.json", "adapter_model.safetensors", "adapter_model.bin"])

def load_tokenizer(model_name: str = BASE_MODEL):
    log(f"Loading tokenizer for {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    log("Tokenizer loaded.")
    return tok

def load_hf_model(model_source: str):
    load_kwargs = dict(
        low_cpu_mem_usage=True,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE_COUNT > 0 else None,
        max_memory=MAX_MEMORY if DEVICE_COUNT > 0 else None,
        attn_implementation="eager",
        trust_remote_code=True,
    )
    load_kwargs = {k: v for k, v in load_kwargs.items() if v is not None}

    # First attempt: sanitized config.
    try:
        with timer(f"HF load: {model_source}"):
            cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
            cfg = sanitize_rope_scaling(cfg)
            model = AutoModelForCausalLM.from_pretrained(model_source, config=cfg, **load_kwargs)
    except Exception as e1:
        log(f"HF load failed for {model_source}: {type(e1).__name__}: {e1}")
        # Second attempt: force rope_scaling removal and try again.
        try:
            with timer(f"HF reload (forced rope fix): {model_source}"):
                cfg = AutoConfig.from_pretrained(model_source, trust_remote_code=True)
                cfg.rope_scaling = None
                model = AutoModelForCausalLM.from_pretrained(model_source, config=cfg, **load_kwargs)
        except Exception as e2:
            log(f"Forced rope-fix load failed for {model_source}: {type(e2).__name__}: {e2}")
            raise

    model.eval()
    try:
        model.config.use_cache = False
    except Exception:
        pass
    return model

def merge_adapter_if_needed(base_model, adapter_path: str):
    if has_adapter(adapter_path) and HAS_PEFT:
        try:
            with timer(f"Merging adapter: {adapter_path}"):
                merged = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
                merged.eval()
            return merged
        except Exception as e:
            log(f"Adapter merge failed; using base model only. {type(e).__name__}: {e}")
    return base_model

def load_variant_model(source: str, tag: str):
    log(f"[{tag}] Loading model from: {source}")
    # If source is a full HF model dir, load it directly.
    if os.path.isdir(source) and os.path.exists(os.path.join(source, "config.json")) and not has_adapter(source):
        hf_model = load_hf_model(source)
    else:
        with timer(f"[{tag}] load base model for adapter merge"):
            base = load_hf_model(BASE_MODEL)
        hf_model = merge_adapter_if_needed(base, source)

    hf_model.eval()
    log(f"[{tag}] Model ready.")
    return hf_model

# ============================================================
# GENERATION / RESIDUAL STREAM EXTRACTION
# ============================================================

@torch.no_grad()
def generate_completion(model, tokenizer, prompt: str, task: str):
    log(f"Generating for task={task} (max_new_tokens={MAX_NEW_TOKENS[task]})")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model_device(model)) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS[task],
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    full_output = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion, inputs["input_ids"], out

def run_forward(model, tokenizer, prompt: str):
    log("Running HF forward with hidden states ...")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model_device(model)) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
    return inputs["input_ids"], out.logits, out

def get_resid_stack(out) -> List[torch.Tensor]:
    return [h[0, -1].detach().float().cpu() for h in out.hidden_states]

def final_norm_and_head(model):
    core = getattr(model, "model", model)
    norm = getattr(core, "norm", None)
    if norm is None:
        norm = getattr(core, "final_layer_norm", None)
    if norm is None:
        norm = getattr(model, "norm", None)
    head = getattr(model, "lm_head", None)
    if head is None:
        head = getattr(model, "embed_out", None)
    if head is None:
        raise RuntimeError("Could not locate final norm / lm_head.")
    return norm, head

def project_resid_to_logits(model, resid_vec: torch.Tensor):
    norm, head = final_norm_and_head(model)
    x = resid_vec.clone().detach().float()
    if x.ndim == 1:
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.ndim == 2:
        x = x.unsqueeze(0)
    with torch.no_grad():
        if norm is not None:
            x = norm(x)
        logits = head(x)
    if logits.ndim == 3:
        logits = logits[0, -1]
    else:
        logits = logits[0]
    return logits.detach().float().cpu()

def cosine_to_final(resid_stack: List[torch.Tensor]) -> List[float]:
    final = resid_stack[-1]
    out = []
    for r in resid_stack:
        num = float(torch.dot(r, final).item())
        den = float(torch.norm(r).item() * torch.norm(final).item() + 1e-12)
        out.append(num / den)
    return out

def layer_norms(resid_stack: List[torch.Tensor]) -> List[float]:
    return [float(torch.norm(r).item()) for r in resid_stack]

# ============================================================
# SAMPLE BUILDING
# ============================================================

def build_samples() -> List[Dict[str, Any]]:
    samples = []

    gsm, gsm_repo = load_gsm8k_dataset()
    gsm, gsm_idx = sample_dataset(gsm, N_GSM8K, SEED, "gsm8k_test")
    for i, s in enumerate(gsm):
        gold = canonical_number(s["answer"].split("####")[-1].strip())
        samples.append({
            "sample_id": f"gsm8k_{i}",
            "task": "gsm8k",
            "subject": "gsm8k",
            "question": s["question"],
            "gold": gold,
            "prompt": build_task_prompt({"task": "gsm8k", "question": s["question"]}),
            "source_index": int(gsm_idx[i]),
            "source_repo": gsm_repo,
        })

    qa = load_strategyqa_dataset()
    qa, qa_idx = sample_dataset(qa, N_STRATEGYQA, SEED, "strategyqa_test")
    for i, s in enumerate(qa):
        gold = "yes" if bool(s["answer"]) else "no"
        samples.append({
            "sample_id": f"strategyqa_{i}",
            "task": "strategyqa",
            "subject": "strategyqa",
            "question": s["question"],
            "gold": gold,
            "prompt": build_task_prompt({"task": "strategyqa", "question": s["question"]}),
            "source_index": int(qa_idx[i]),
        })

    mmlu_candidates = []
    for subject in MMLU_SUBJECTS:
        try:
            ds, split_used = load_mmlu_subject(subject)
        except Exception:
            continue
        ds, idx = sample_dataset(ds, 1, SEED, f"mmlu_{subject}_{split_used}")
        for i, s in enumerate(ds):
            choices = get_mmlu_choices(s)
            gold = safe_mmlu_gold(s)
            if len(choices) >= 4 and gold is not None:
                mmlu_candidates.append({
                    "sample_id": f"mmlu_{subject}_{i}",
                    "task": "mmlu",
                    "subject": subject,
                    "question": s["question"],
                    "choices": choices[:4],
                    "gold": gold,
                    "prompt": build_task_prompt({"task": "mmlu", "question": s["question"], "choices": choices[:4]}),
                    "source_index": int(idx[i]),
                    "split_used": split_used,
                })
    if len(mmlu_candidates) > N_MMLU:
        random.shuffle(mmlu_candidates)
        mmlu_candidates = mmlu_candidates[:N_MMLU]
    samples.extend(mmlu_candidates)

    random.shuffle(samples)
    return samples

# ============================================================
# PARSING / METRICS
# ============================================================

def parse_prediction(sample: Dict[str, Any], completion: str):
    if sample["task"] == "gsm8k":
        pred = canonical_number(extract_number(completion))
        return pred, bool(is_number_correct(pred, sample["gold"]))
    if sample["task"] == "strategyqa":
        pred = extract_yes_no(completion)
        return pred, bool(pred == sample["gold"])
    if sample["task"] == "mmlu":
        pred = extract_mcq(completion)
        return pred, bool(pred == sample["gold"])
    return None, False

def tensor_prob_for_ids(logits: torch.Tensor, ids: List[int]) -> float:
    if len(ids) == 0:
        return float("nan")
    probs = torch.softmax(logits.float(), dim=-1)
    return float(max([probs[i].item() for i in ids]))

def tensor_logit_for_ids(logits: torch.Tensor, ids: List[int]) -> float:
    if len(ids) == 0:
        return float("nan")
    return float(max([logits[i].item() for i in ids]))

def analyze_sample(model, tokenizer, sample: Dict[str, Any], model_tag: str):
    prompt = sample["prompt"]

    log(f"Generating sample {sample['sample_id']} ({sample['task']}) ...")
    full_output, completion, token_ids, gen_out = generate_completion(model, tokenizer, prompt, sample["task"])
    pred, correct = parse_prediction(sample, completion)

    log(f"Forward pass for {sample['sample_id']} ...")
    tokens, logits, out = run_forward(model, tokenizer, prompt)
    resid_stack = get_resid_stack(out)
    if len(resid_stack) == 0:
        raise RuntimeError(f"No residual stream vectors extracted for {sample['sample_id']}")

    final_logits = logits[0, -1].detach().float().cpu()
    final_top = topn_token_strings(tokenizer, final_logits, n=5)
    final_entropy = float(-(torch.softmax(final_logits, dim=-1) * torch.log_softmax(final_logits, dim=-1)).sum().item())

    cand = candidate_tokens(tokenizer, sample, final_logits, pred)
    cand_ids = {c: token_ids_for_text(tokenizer, c) for c in cand}

    norms = layer_norms(resid_stack)
    cosines = cosine_to_final(resid_stack)
    layer_rows = []
    for layer_idx, resid in enumerate(resid_stack):
        layer_logits = project_resid_to_logits(model, resid)
        top_tok, top_prob = topn_token_strings(tokenizer, layer_logits, n=1)[0]
        row = {
            "model_tag": model_tag,
            "sample_id": sample["sample_id"],
            "task": sample["task"],
            "subject": sample["subject"],
            "layer": layer_idx,
            "resid_norm": norms[layer_idx],
            "cosine_to_final": cosines[layer_idx],
            "top_token": top_tok,
            "top_prob": top_prob,
            "entropy": float(-(torch.softmax(layer_logits, dim=-1) * torch.log_softmax(layer_logits, dim=-1)).sum().item())),
        }
        for c, ids in cand_ids.items():
            row[f"prob_{c}"] = tensor_prob_for_ids(layer_logits, ids)
            row[f"logit_{c}"] = tensor_logit_for_ids(layer_logits, ids)
        layer_rows.append(row)
    layer_df = pd.DataFrame(layer_rows)

    target_ids = {
        "gold": token_ids_for_text(tokenizer, str(sample["gold"])),
        "pred": token_ids_for_text(tokenizer, str(pred or sample["gold"])),
    }

    summary = {
        "model_tag": model_tag,
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "subject": sample["subject"],
        "question": sample["question"],
        "gold": sample["gold"],
        "prediction": pred,
        "correct": bool(correct),
        "prompt": prompt,
        "full_output": full_output,
        "completion": completion,
        "completion_len": len(tokenizer.encode(completion, add_special_tokens=False)) if hasattr(tokenizer, "encode") else len(completion.split()),
        "prompt_len": int(tokens.shape[1]) if hasattr(tokens, "shape") else 0,
        "final_top_token": final_top[0][0],
        "final_top_prob": final_top[0][1],
        "final_entropy": final_entropy,
        "hedged": bool(re.search(r"\b(maybe|probably|perhaps|uncertain|not sure|depends)\b", completion.lower())),
        "format_ok": bool(explicit_answer_present(completion)),
        "candidate_tokens": json.dumps(cand),
        "gold_prob_final": tensor_prob_for_ids(final_logits, target_ids["gold"]),
        "pred_prob_final": tensor_prob_for_ids(final_logits, target_ids["pred"]),
        "gold_logit_final": tensor_logit_for_ids(final_logits, target_ids["gold"]),
        "pred_logit_final": tensor_logit_for_ids(final_logits, target_ids["pred"]),
        "n_layers": len(resid_stack),
    }

    log(f"Sample {sample['sample_id']} done | correct={correct} | pred={pred} | gold={sample['gold']}")
    return summary, layer_df, resid_stack, cand, final_logits

# ============================================================
# VISUALISATIONS
# ============================================================

def plot_residual_dashboard(sample_summary: Dict[str, Any], layer_df: pd.DataFrame, resid_stack: List[torch.Tensor], candidates: List[str], final_logits: torch.Tensor, model_tag: str, out_path: str):
    log(f"Rendering residual dashboard for {sample_summary['sample_id']} ({model_tag}) ...")
    layer_df = layer_df.sort_values("layer")
    layers = layer_df["layer"].tolist()
    cand_cols = [f"prob_{c}" for c in candidates if f"prob_{c}" in layer_df.columns]
    heat = layer_df[cand_cols].to_numpy(dtype=np.float64).T if len(cand_cols) else np.zeros((1, len(layers)))

    X = np.stack([r.numpy() for r in resid_stack], axis=0)
    Z, _ = np_pca_2d(X)

    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(layers, layer_df["resid_norm"], marker="o", linewidth=2.2, label="residual norm")
    ax1.set_title(f"{model_tag.upper()} | {sample_summary['task']} | residual stream")
    ax1.set_xlabel("Layer")
    ax1.set_ylabel("L2 norm")
    ax1b = ax1.twinx()
    ax1b.plot(layers, layer_df["cosine_to_final"], marker="s", linewidth=2.0, color="tab:orange", label="cosine to final")
    ax1b.set_ylabel("Cosine to final")
    ax1.legend(loc="upper left", frameon=True)
    ax1b.legend(loc="lower right", frameon=True)

    ax2 = fig.add_subplot(gs[0, 1])
    if len(cand_cols) > 0:
        im = ax2.imshow(heat, aspect="auto", cmap="viridis", origin="lower")
        ax2.set_yticks(range(len(candidates)))
        ax2.set_yticklabels(candidates)
        ax2.set_xticks(range(len(layers)))
        ax2.set_xticklabels(layers)
        ax2.set_title("Logit lens: candidate probability by layer")
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    else:
        ax2.text(0.5, 0.5, "No candidate heatmap available", ha="center", va="center")
        ax2.axis("off")

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(Z[:, 0], Z[:, 1], marker="o", linewidth=2.2)
    for i, (x, y) in enumerate(Z):
        ax3.annotate(str(i), (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)
    ax3.set_title("Residual stream PCA trajectory")
    ax3.set_xlabel("PC1")
    ax3.set_ylabel("PC2")

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")
    rows = [
        ["Gold", str(sample_summary["gold"])],
        ["Prediction", str(sample_summary["prediction"])],
        ["Correct", str(sample_summary["correct"])],
        ["Final top token", str(sample_summary["final_top_token"])],
        ["Final entropy", f"{sample_summary['final_entropy']:.3f}"],
        ["Final gold prob", f"{sample_summary['gold_prob_final']:.4f}"],
        ["Final pred prob", f"{sample_summary['pred_prob_final']:.4f}"],
        ["Format ok", str(sample_summary["format_ok"])],
    ]
    table = ax4.table(cellText=rows, colLabels=["Metric", "Value"], loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.6)
    ax4.set_title("Prediction summary")

    plt.suptitle(f"Residual stream dashboard | {sample_summary['sample_id']} | {model_tag}", fontsize=16, y=1.02)
    save_plot(out_path)

    if HAS_PLOTLY:
        try:
            figp = go.Figure()
            figp.add_trace(go.Scatter(
                x=Z[:, 0],
                y=Z[:, 1],
                mode="lines+markers+text",
                text=[str(i) for i in range(len(Z))],
                textposition="top center",
                name="trajectory"
            ))
            figp.update_layout(
                title=f"Residual stream PCA trajectory | {sample_summary['sample_id']} | {model_tag}",
                xaxis_title="PC1",
                yaxis_title="PC2",
                width=1000,
                height=700,
            )
            figp.write_html(out_path.replace(".png", ".html"))
        except Exception as e:
            log(f"Plotly HTML export failed for {sample_summary['sample_id']}: {type(e).__name__}: {e}")

def plot_comparison_dashboard(
    base_item: Dict[str, Any],
    sft_item: Dict[str, Any],
    base_layer_df: pd.DataFrame,
    sft_layer_df: pd.DataFrame,
    base_resid: List[torch.Tensor],
    sft_resid: List[torch.Tensor],
    out_path: str,
):
    log(f"Rendering comparison dashboard for {base_item['sample_id']} ...")
    base_layer_df = base_layer_df.sort_values("layer")
    sft_layer_df = sft_layer_df.sort_values("layer")
    base_layers = base_layer_df["layer"].tolist()
    sft_layers = sft_layer_df["layer"].tolist()

    X = np.stack([r.numpy() for r in base_resid + sft_resid], axis=0)
    Z, _ = np_pca_2d(X)
    Z_base = Z[: len(base_resid)]
    Z_sft = Z[len(base_resid) :]

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 2)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(base_layers, base_layer_df["resid_norm"], marker="o", linewidth=2.2, label="base")
    ax1.plot(sft_layers, sft_layer_df["resid_norm"], marker="s", linewidth=2.2, label="sft")
    ax1.set_title("Residual norm overlay")
    ax1.set_xlabel("Layer")
    ax1.set_ylabel("L2 norm")
    ax1.legend(frameon=True)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(base_layers, base_layer_df["cosine_to_final"], marker="o", linewidth=2.2, label="base")
    ax2.plot(sft_layers, sft_layer_df["cosine_to_final"], marker="s", linewidth=2.2, label="sft")
    ax2.set_title("Cosine to final overlay")
    ax2.set_xlabel("Layer")
    ax2.set_ylabel("Cosine")
    ax2.legend(frameon=True)

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(Z_base[:, 0], Z_base[:, 1], marker="o", linewidth=2.2, label="base")
    ax3.plot(Z_sft[:, 0], Z_sft[:, 1], marker="s", linewidth=2.2, label="sft")
    for i, (x, y) in enumerate(Z_base):
        ax3.annotate(f"B{i}", (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)
    for i, (x, y) in enumerate(Z_sft):
        ax3.annotate(f"S{i}", (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)
    ax3.set_title("Residual stream PCA comparison")
    ax3.set_xlabel("PC1")
    ax3.set_ylabel("PC2")
    ax3.legend(frameon=True)

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")
    rows = [
        ["Task", base_item["task"]],
        ["Gold", str(base_item["gold"])],
        ["Base pred", str(base_item["prediction"])],
        ["SFT pred", str(sft_item["prediction"])],
        ["Base correct", str(base_item["correct"])],
        ["SFT correct", str(sft_item["correct"])],
        [
            "Delta final resid norm",
            f"{float(sft_layer_df['resid_norm'].iloc[-1] - base_layer_df['resid_norm'].iloc[-1]):.4f}",
        ],
        [
            "Delta final cosine",
            f"{float(sft_layer_df['cosine_to_final'].iloc[-1] - base_layer_df['cosine_to_final'].iloc[-1]):.4f}",
        ],
    ]
    table = ax4.table(cellText=rows, colLabels=["Metric", "Value"], loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.6)
    ax4.set_title("Comparison summary")

    plt.suptitle(
        f"Base vs SFT residual stream comparison | {base_item['sample_id']}",
        fontsize=16,
        y=1.02,
    )
    save_plot(out_path)

    if HAS_PLOTLY:
        try:
            figp = go.Figure()
            figp.add_trace(
                go.Scatter(
                    x=Z_base[:, 0],
                    y=Z_base[:, 1],
                    mode="lines+markers+text",
                    text=[f"B{i}" for i in range(len(Z_base))],
                    name="base",
                )
            )
            figp.add_trace(
                go.Scatter(
                    x=Z_sft[:, 0],
                    y=Z_sft[:, 1],
                    mode="lines+markers+text",
                    text=[f"S{i}" for i in range(len(Z_sft))],
                    name="sft",
                )
            )
            figp.update_layout(
                title=f"Base vs SFT residual stream PCA comparison | {base_item['sample_id']}",
                xaxis_title="PC1",
                yaxis_title="PC2",
                width=1000,
                height=700,
            )
            figp.write_html(out_path.replace(".png", ".html"))
        except Exception as e:
            log(f"Plotly HTML export failed for {base_item['sample_id']}: {type(e).__name__}: {e}")

# ============================================================
# ANALYSIS
# ============================================================

def analyze_model(model, tokenizer, samples: List[Dict[str, Any]], model_tag: str):
    sample_rows = []
    layer_rows = []
    artifacts = {}
    log(f"[{model_tag}] Starting analysis for {len(samples)} samples.")
    gpu_report(f"[{model_tag}] pre-analysis GPU")

    for i, sample in enumerate(samples):
        log(f"[{model_tag}] {i+1}/{len(samples)} | {sample['task']} | {preview_text(sample['question'], 90)}")
        summary, layer_df, resid_stack, cand, final_logits = analyze_sample(model, tokenizer, sample, model_tag)
        sample_rows.append(summary)
        layer_rows.append(layer_df)
        artifacts[sample["sample_id"]] = {
            "summary": summary,
            "layer_df": layer_df,
            "resid_stack": resid_stack,
            "cand": cand,
            "final_logits": final_logits,
        }

        task_dir = os.path.join(PLOT_DIR, model_tag)
        os.makedirs(task_dir, exist_ok=True)
        plot_residual_dashboard(summary, layer_df, resid_stack, cand, final_logits, model_tag, os.path.join(task_dir, f"{sample['sample_id']}_dashboard.png"))
        save_df(layer_df, os.path.join(CSV_DIR, f"{model_tag}_{sample['sample_id']}_layerwise.csv"))
        gpu_report(f"[{model_tag}] after sample {i+1}")
        free_memory()

    sample_df = pd.DataFrame(sample_rows)
    layer_df = pd.concat(layer_rows, ignore_index=True) if layer_rows else pd.DataFrame()
    log(f"[{model_tag}] Analysis complete.")
    return sample_df, layer_df, artifacts

def plot_global_summary(base_df: pd.DataFrame, sft_df: pd.DataFrame):
    base_sum = base_df.groupby("task", as_index=False).agg(
        n=("correct", "count"),
        accuracy=("correct", "mean"),
        format_rate=("format_ok", "mean"),
        hedge_rate=("hedged", "mean"),
        avg_len=("completion_len", "mean"),
    )
    sft_sum = sft_df.groupby("task", as_index=False).agg(
        n=("correct", "count"),
        accuracy=("correct", "mean"),
        format_rate=("format_ok", "mean"),
        hedge_rate=("hedged", "mean"),
        avg_len=("completion_len", "mean"),
    )
    comp = base_sum.merge(sft_sum, on="task", suffixes=("_base", "_sft"))
    comp["delta_accuracy"] = comp["accuracy_sft"] - comp["accuracy_base"]
    comp["delta_format_rate"] = comp["format_rate_sft"] - comp["format_rate_base"]
    comp["delta_hedge_rate"] = comp["hedge_rate_sft"] - comp["hedge_rate_base"]

    save_df(base_sum, os.path.join(CSV_DIR, "base_summary.csv"))
    save_df(sft_sum, os.path.join(CSV_DIR, "sft_summary.csv"))
    save_df(comp, os.path.join(CSV_DIR, "comparison_table.csv"))

    x = np.arange(len(comp))
    w = 0.35
    plt.figure(figsize=(9.2, 5.0))
    plt.bar(x - w / 2, comp["accuracy_base"], width=w, label="base")
    plt.bar(x + w / 2, comp["accuracy_sft"], width=w, label="sft")
    plt.xticks(x, comp["task"], rotation=20)
    plt.ylabel("Accuracy")
    plt.title("Base vs SFT accuracy")
    plt.legend(frameon=True)
    for i, v in enumerate(comp["accuracy_base"]):
        plt.text(i - w / 2, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    for i, v in enumerate(comp["accuracy_sft"]):
        plt.text(i + w / 2, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    save_plot(os.path.join(PLOT_DIR, "accuracy_comparison.png"))

    return comp, base_sum, sft_sum

# ============================================================
# MAIN
# ============================================================

def main():
    print("=======================================================")
    print("PHI-3 RESIDUAL STREAM DASHBOARD (HF-SAFE VERSION)")
    print("=======================================================")
    log("Boot sequence started.")
    gpu_report("Boot GPU")

    with timer("Tokenizer load"):
        tokenizer = load_tokenizer(BASE_MODEL)

    with timer("Dataset building"):
        samples = build_samples()
    if len(samples) == 0:
        raise RuntimeError("No samples were loaded.")

    save_df(pd.DataFrame(samples), os.path.join(CSV_DIR, "selected_samples.csv"))
    log(f"Selected {len(samples)} samples: {[s['sample_id'] for s in samples]}")
    gpu_report("Post-dataset GPU")

    with timer("Base model load"):
        base_model = load_variant_model(BASE_MODEL, "BASE")
    gpu_report("After base load")
    with timer("Base model analysis"):
        base_df, base_layer_df, base_artifacts = analyze_model(base_model, tokenizer, samples, "base")
    save_df(base_df, os.path.join(CSV_DIR, "base_sample_summary.csv"))
    save_df(base_layer_df, os.path.join(CSV_DIR, "base_layerwise.csv"))
    save_json({k: v["summary"] for k, v in base_artifacts.items()}, os.path.join(REPORT_DIR, "base_sample_summaries.json"))
    free_memory(base_model)

    with timer("SFT model load"):
        sft_model = load_variant_model(SFT_PATH, "SFT")
    gpu_report("After SFT load")
    with timer("SFT model analysis"):
        sft_df, sft_layer_df, sft_artifacts = analyze_model(sft_model, tokenizer, samples, "sft")
    save_df(sft_df, os.path.join(CSV_DIR, "sft_sample_summary.csv"))
    save_df(sft_layer_df, os.path.join(CSV_DIR, "sft_layerwise.csv"))
    save_json({k: v["summary"] for k, v in sft_artifacts.items()}, os.path.join(REPORT_DIR, "sft_sample_summaries.json"))
    free_memory(sft_model)

    with timer("Global summary plotting"):
        comp, base_sum, sft_sum = plot_global_summary(base_df, sft_df)

    compare_dir = os.path.join(PLOT_DIR, "comparison")
    os.makedirs(compare_dir, exist_ok=True)
    for sample in samples:
        sid = sample["sample_id"]
        if sid not in base_artifacts or sid not in sft_artifacts:
            continue
        log(f"Rendering comparison dashboard for {sid} ...")
        plot_comparison_dashboard(
            base_artifacts[sid]["summary"],
            sft_artifacts[sid]["summary"],
            base_artifacts[sid]["layer_df"],
            sft_artifacts[sid]["layer_df"],
            base_artifacts[sid]["resid_stack"],
            sft_artifacts[sid]["resid_stack"],
            os.path.join(compare_dir, f"{sid}_comparison.png"),
        )

    merged_layers = base_layer_df.merge(
        sft_layer_df,
        on=["model_tag", "sample_id", "task", "subject", "layer"],
        how="outer",
        suffixes=("_base", "_sft"),
    )
    save_df(merged_layers, os.path.join(CSV_DIR, "merged_layerwise.csv"))
    gpu_report("Before report write")

    report_lines = [
        "# Phi-3 residual stream report\n",
        f"- Base model: `{BASE_MODEL}`",
        f"- SFT source: `{SFT_PATH}`",
        f"- Samples per benchmark: GSM8K={N_GSM8K}, StrategyQA={N_STRATEGYQA}, MMLU={N_MMLU}",
        "\n## Base summary\n",
        base_sum.to_markdown(index=False),
        "\n## SFT summary\n",
        sft_sum.to_markdown(index=False),
        "\n## Comparison\n",
        comp.to_markdown(index=False),
        "\n## Notes\n",
        "- Residual stream representations are visualised for two random questions per benchmark total.",
        "- The code uses HF hidden states directly because the prior TransformerLens cache path killed the kernel.",
        "- GSM8K uses gsm8k first and openai/gsm8k / madrylab/gsm8k-platinum as fallbacks.",
        "- The code loads only one model at a time to avoid Kaggle OOMs.",
        "- RoPE scaling is sanitized before Phi-3 model construction to prevent the \"Unknown RoPE scaling type default\" crash.",
        "- Verbose timestamps and GPU memory reports are included around expensive steps.",
    ]
    with open(os.path.join(REPORT_DIR, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    log(f"Saved report -> {os.path.join(REPORT_DIR, 'report.md')}")

    save_json(
        {
            "base_model": BASE_MODEL,
            "sft_source": SFT_PATH,
            "samples": [s["sample_id"] for s in samples],
            "base_summary": base_sum.to_dict(orient="records"),
            "sft_summary": sft_sum.to_dict(orient="records"),
            "comparison": comp.to_dict(orient="records"),
        },
        os.path.join(REPORT_DIR, "summary.json"),
    )

    print("\n===== BASE SUMMARY =====")
    print(base_sum.to_string(index=False))
    print("\n===== SFT SUMMARY =====")
    print(sft_sum.to_string(index=False))
    print("\n===== COMPARISON =====")
    print(comp.to_string(index=False))
    print("\nSaved outputs to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOT_DIR}")
    print(f"- HTML: {HTML_DIR}")
    print(f"- Reports: {REPORT_DIR}")

if __name__ == "__main__":
    main()

: 